In [2]:
%pip install pandas
%pip install numpy 

You should consider upgrading via the '/Users/niranjanhebli/Desktop/Week3/week3/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/niranjanhebli/Desktop/Week3/week3/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import re

### Dataset Summary

In [ ]:
# Data Profiling Functions
def dataset_summary(df: pd.DataFrame) -> dict:
    """ Returns a summary of the dataset including number of rows, columns, and total null values."""
    return {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "total_nulls": int(df.isna().sum().sum())
    }

### Basic Column Summary

In [ ]:
# Column Profiling Functions
def basic_column_stats(series: pd.Series) -> dict:
    """ Returns basic statistics for a given column including data type, unique values, null count, and top values."""

    null_count = series.isna().sum()

    return {
        "dtype": str(series.dtype),
        "unique_count": int(series.nunique(dropna=True)),
        "null_count": int(null_count),
        "null_percentage": round((null_count / len(series)) * 100, 2),
        "top_values": series.value_counts(dropna=False).head(5).to_dict()
    }

### Numeric statistics

In [ ]:
# String Profiling Functions
def numeric_stats(series: pd.Series) -> dict:
    """ Returns statistics for numeric columns including min, max, mean, median, std, skewness, and outlier count."""

    stats = {
        "min": series.min(),
        "max": series.max(),
        "mean": series.mean(),
        "median": series.median(),
        "std": series.std(),
        "skewness": series.skew()
    }

    # Outlier detection (>3 std)
    if series.std() != 0:
        z_scores = (series - series.mean()) / series.std()
        outliers = series[np.abs(z_scores) > 3]
        stats["suspicious_outliers"] = int(outliers.count())
    else:
        stats["suspicious_outliers"] = 0

    return stats


### Pattern detection

In [ ]:
# String Pattern Detection Functions
def detect_patterns(series: pd.Series) -> dict:
    """ Returns counts of values matching specific patterns such as numeric-like, email-like, and containing digits."""

    return {
        "numeric_like": series.str.match(r"^\d+$").sum(),
        "email_like": series.str.contains(r"@", na=False).sum(),
        "contains_digits": series.str.contains(r"\d", na=False).sum()
    }


### String Statistics

In [ ]:
def string_stats(series: pd.Series) -> dict:
 
    """ Returns statistics for string columns including average length, min/max length, and pattern counts."""

    clean_series = series.dropna().astype(str)

    if len(clean_series) == 0:
        return {}

    lengths = clean_series.apply(len)

    return {
        "avg_length": lengths.mean(),
        "min_length": lengths.min(),
        "max_length": lengths.max(),
        "patterns": detect_patterns(clean_series)
    }

### Issue detection

In [ ]:
def detect_column_issues(series: pd.Series) -> list:
    """ Detects potential issues in a column such as single value, high cardinality, and high missing values."""

    issues = []

    unique_count = series.nunique(dropna=True)
    null_pct = (series.isna().sum() / len(series)) * 100

    if unique_count == 1:
        issues.append("single_value_column")

    if unique_count > len(series) * 0.8 and series.dtype == "object":
        issues.append("high_cardinality")

    if null_pct > 50:
        issues.append("high_missing_values")

    return issues

### Profile single column

In [ ]:
def profile_column(series: pd.Series) -> dict:
    """ Profiles a single column and returns a dictionary of statistics and potential issues."""

    profile = basic_column_stats(series)

    if pd.api.types.is_numeric_dtype(series):
        profile["numeric_stats"] = numeric_stats(series)

    if pd.api.types.is_object_dtype(series):
        profile["string_stats"] = string_stats(series)

    profile["potential_issues"] = detect_column_issues(series)

    return profile


### Print Summary

In [ ]:
def print_summary(df: pd.DataFrame, profile: dict):
    """ Prints a summary of the dataset and column profiles in a readable format."""

    print("\n===== DATASET SUMMARY =====")
    print(profile["dataset_summary"])

    print("\n===== COLUMN SUMMARY =====")

    for col, info in profile["columns"].items():

        print(f"\nColumn: {col}")
        print(f"dtype: {info['dtype']}")
        print(f"unique: {info['unique_count']}")
        print(f"nulls: {info['null_count']} ({info['null_percentage']}%)")

        print("top values:", list(info["top_values"].items())[:3])

        if "numeric_stats" in info:
            stats = info["numeric_stats"]
            print(f"mean: {round(stats['mean'],2)} | std: {round(stats['std'],2)}")

        if "string_stats" in info:
            s = info["string_stats"]
            print(f"avg length: {round(s['avg_length'],2)}")

        if info["potential_issues"]:
            print("issues:", info["potential_issues"])

### Profile DataFrame

In [ ]:
def profile_dataframe(df: pd.DataFrame) -> dict:
    """ Profiles a pandas DataFrame and returns a dictionary of column profiles."""

    profile = {
        "dataset_summary": dataset_summary(df),
        "columns": {}
    }

    for col in df.columns:
        profile["columns"][col] = profile_column(df[col])

    print_summary(df, profile)

    return profile

In [15]:

data = {
"age": [25, 30, 35, 40, None, 120, 25, 25],
"name": ["Alice", "Bob", "Charlie", "David", "Eve", "Frank", "Alice", None],
"salary": [50000, 60000, 70000, 80000, 90000, 100000, 50000, 50000],
"email": ["a@test.com", "b@test.com", None, "d@test.com", "e@test.com", "f@test.com", "a@test.com", "x"]
}

df = pd.DataFrame(data)

profile = profile_dataframe(df)


===== DATASET SUMMARY =====
{'rows': 8, 'columns': 4, 'total_nulls': 3}

===== COLUMN SUMMARY =====

Column: age
dtype: float64
unique: 5
nulls: 1 (12.5%)
top values: [(25.0, 3), (30.0, 1), (35.0, 1)]
mean: 42.86 | std: 34.5

Column: name
dtype: object
unique: 6
nulls: 1 (12.5%)
top values: [('Alice', 2), ('Bob', 1), ('Charlie', 1)]
avg length: 4.71

Column: salary
dtype: int64
unique: 6
nulls: 0 (0.0%)
top values: [(50000, 3), (60000, 1), (70000, 1)]
mean: 68750.0 | std: 19594.1

Column: email
dtype: object
unique: 6
nulls: 1 (12.5%)
top values: [('a@test.com', 2), ('b@test.com', 1), (None, 1)]
avg length: 8.71
